In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

# General

In [2]:
from ioutils import FileIO
from fileutils import DirInfo, FileInfo
from master import MasterParams
from pandas import Series, DataFrame
from sys import prefix
from listUtils import getFlatList
from musicdb import MusicDBIO
from utils import MusicDBPermDir
mp    = MasterParams(verbose=True)
io    = FileIO()
mdbpd = MusicDBPermDir()

MasterParams()
  ==> DBs:       ['Discogs', 'Spotify', 'LastFM', 'Genius', 'RateYourMusic', 'Deezer', 'AllMusic', 'MusicBrainz', 'AlbumOfTheYear']
  ==> Raw Path:  /Volumes/Piggy/Discog
  ==> Mod Path:  /Volumes/Seagate/Discog
  ==> Sum Path:  /Users/tgadfort/Music/Discog
  ==> MaxModVal: 100
  ==> Project:   pandb
  ==> MusicDB:   musicdb


# DB-Specific

In [3]:
from lib import deezer
mio   = deezer.MusicDBIO(verbose=False)
apiio = deezer.RawAPIData()
db    = mio.db
permDBDir = mdbpd.getDBPermPath("Deezer")
print("Saving Perminant {0} DB Data To {1}".format(db, permDBDir.str))

Deezer API(Key=None)
Saving Perminant Deezer DB Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdblib/Deezer


# Master Files

In [4]:
from base import MusicDBDir, MusicDBData
permDir = MusicDBDir(permDBDir)
localRelatedData        = MusicDBData(path=permDir, fname="{0}SearchedForLocalRelatedData".format(db.lower()))
localRelated            = MusicDBData(path=permDir, fname="{0}SearchedForLocalRelated".format(db.lower()))
masterRelatedArtistData = mio.data.getRelatedArtistsData()
localArtistsInfoData    = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtistsInfoData".format(db.lower()))
localArtistsInfo        = MusicDBData(path=permDir, fname="{0}SearchedForLocalArtistsInfo".format(db.lower()))
masterArtistsInfoData   = mio.data.getArtistsInfoData()
knownArtists            = mio.data.getSummaryNameData()
errors                  = MusicDBData(path=permDir, fname="{0}SearchedForErrors".format(db.lower()))

In [5]:
##########################################################################################
# Show Summary
##########################################################################################
print("{0} Search Results".format(db))
print("   Local Related Artists:       {0}".format(len(localRelated.get())))
print("   Local Related Artists Data:  {0}".format(len(localRelatedData.get())))
print("   Master Related Artists Data: {0}".format(len(masterRelatedArtistData)))
print("   Local Artists Info:          {0}".format(len(localArtistsInfo.get())))
print("   Local Artists Info Data:     {0}".format(len(localArtistsInfoData.get())))
print("   Master Artists Info Data:    {0}".format(masterArtistsInfoData.shape[0]))
print("   Errors:                      {0}".format(len(errors.get())))
print("   Known Summary IDs:           {0}".format(len(knownArtists)))

Deezer Search Results
   Local Related Artists:       93208
   Local Related Artists Data:  0
   Master Related Artists Data: 62431
   Local Artists Info:          58469
   Local Artists Info Data:     0
   Master Artists Info Data:    58469
   Errors:                      1781
   Known Summary IDs:           1069560


# Download Artist Info Data

In [6]:
mio   = deezer.MusicDBIO(verbose=False)
apiio = deezer.RawAPIData(debug=True)

Deezer API(Key=None)


## Find Artist IDs To Get

In [14]:
artistNames = mio.data.getSummaryNameData()
artistNames.name = "ArtistName"
basicData = DataFrame(artistNames).join(mio.data.getSummaryNumAlbumsData()).sort_values(by="NumAlbums", ascending=False)
localArtistsInfoDict = localArtistsInfo.get()
artistIDsToGet = basicData[~basicData.index.isin(localArtistsInfoDict.keys())]["ArtistName"]

print("{0} Search Results".format(db))
print("   Known Master Basic Data:   {0}".format(len(artistNames)))
print("   Known Artist Info Names:   {0}".format(len(localArtistsInfoDict)))
print("   Artist Names To Get:       {0}".format(len(artistIDsToGet)))

#   Artist Names To Get:       1020553

Deezer Search Results
   Known Master Basic Data:   1069560
   Known Artist Info Names:   58519
   Artist Names To Get:       1020503


In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("today", "7:36pm")
tt = TermTime("today", "11:00pm")
maxN = 50000000

n  = 0
searchedForLocalArtistsInfo     = localArtistsInfo.get()
searchedForLocalArtistsInfoData = localArtistsInfoData.get()
searchedForErrors               = errors.get()
for i,(artistID,artistName) in enumerate(artistIDsToGet.iteritems()):
    if searchedForLocalArtistsInfo.get(artistID) is not None:
        continue

    response = apiio.getArtistInfoData(artistName=artistName, artistID=artistID)
    if response is None or len(response) == 0:
        if response is None:
            print("Error ==> {0}".format((artistID,artistName)))
            searchedForErrors[artistID] = artistName
            apiio.sleep(3.5)
        else:
            searchedForLocalArtistsInfo[artistID] = "NoInfo"
            apiio.sleep(1.5)
        continue
    
    searchedForLocalArtistsInfo[artistID]     = artistName
    searchedForLocalArtistsInfoData[artistID] = response
    apiio.sleep(1.5)
    n += 1
        
    if n % 25 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(searchedForLocalArtistsInfo), len(searchedForLocalArtistsInfoData), db))
        localArtistsInfo.save(data=searchedForLocalArtistsInfo)
        localArtistsInfoData.save(data=searchedForLocalArtistsInfoData)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break

ts.stop()
print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(searchedForLocalArtistsInfo), db))
localArtistsInfo.save(data=searchedForLocalArtistsInfo)
localArtistsInfoData.save(data=searchedForLocalArtistsInfoData)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

Starting Process [Getting Deezer ArtistIDs]    ==> Time Is 2022-03-12 21:35:39
========================= termTime(day=today,time=11:00pm) =========================
   ====> Terminate Time Set To 2022-03-12 23:00:00 <====
   ====> Will Terminate Process 1 Hour and 24 Minutes From Now
Searching For Artist Info For The Hillbilly Five (4411686)                      	True
Searching For Artist Info For Outatime (1329837)                                	True
Searching For Artist Info For Damian Wasse (1454564)                            	True
Searching For Artist Info For Mary Travers (203761)                             	True
Searching For Artist Info For Clark Kessinger (302134)                          	True
Searching For Artist Info For Bonnie Dobson (376093)                            	True
Searching For Artist Info For AudioStorm (1641784)                              	True
Searching For Artist Info For Simplex Sensus (6241588)                          	True
Searching For Artist Info Fo

## Save Results

In [11]:
from pandas import DataFrame, Series, concat
from listUtils import getFlatList

def getArtistNamesDataFrame(laid):
    df = None
    if isinstance(laid,dict) and len(laid) > 0:
        df = DataFrame(laid.values())
    return df
        
def getResponseDataFrame(laid):
    df = getArtistNamesDataFrame(laid)
    if not isinstance(df,DataFrame):
        return None
    df.index = df['id']
    df.drop(['id'], axis=1, inplace=True)
    return df

In [12]:
laid = localArtistsInfoData.get()
df  = getResponseDataFrame(laid)
if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = mio.data.getArtistsInfoData()    
    if isinstance(searchDF,DataFrame):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = concat([searchDF,df])
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Artists".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Artists".format(searchDF.shape[0]))
    searchDF['fans'] = searchDF['fans'].astype(int)
    searchDF = searchDF.sort_values(by='fans', ascending=False)
    print("Found {0} Unique + Sorted Artists".format(searchDF.shape[0]))
    print("Saving Data")
    mio.data.saveArtistsInfoData(data=searchDF)
else:
    print("No new artists")

Found 50 New Artists
Found 58469 Previous Artists
Found 58519 Total Artists
Found 58519 Unique Artists
Found 58519 Unique + Sorted Artists
Saving Data


In [13]:
localArtistsInfoData.save(data={})

# Download Related Artist Search Data

In [ ]:
mio   = deezer.MusicDBIO(verbose=False)
apiio = deezer.RawAPIData(debug=True)

## Find Related Artists

In [ ]:
useBasicData       = False
useSelfRelatedData = True
useMasterIDs       = False

if useBasicData is True:
    knownRelatedArtists = localRelated.get()
    basicData = DataFrame(mio.data.getSummaryNameData()).join(mio.data.getSummaryNumAlbumsData()).sort_values(by="NumAlbums", ascending=False)
    basicData.columns = ["ArtistName", "NumAlbums"]
    artistRelatedToGet = basicData["ArtistName"][~basicData["ArtistName"].index.isin(knownRelatedArtists.keys())]

    print("{0} Search Results".format(db))
    print("   Known Master Basic Data:     {0}".format(basicData.shape[0]))
    print("   Known Local Related Names:   {0}".format(len(knownRelatedArtists)))
    print("   Artist Names To Get:         {0}".format(len(artistRelatedToGet)))
elif useSelfRelatedData is True:
    artistRelatedToGet  = {}
    knownRelatedArtists = localRelated.get()
    masterRelatedArtistData = mio.data.getRelatedArtistsData()
    for artistID,artistIDData in masterRelatedArtistData.iteritems():
        artistRelatedToGet.update({str(k): v for k,v in artistIDData.items() if knownRelatedArtists.get(str(k)) is None})
    artistRelatedToGet = Series(artistRelatedToGet)

    print("{0} Search Results".format(db))
    print("   Known Master Related Data:   {0}".format(len(masterRelatedArtistData)))
    print("   Known Local Related Names:   {0}".format(len(knownRelatedArtists)))
    print("   Artist Names To Get:         {0}".format(len(artistRelatedToGet)))
elif useMasterIDs is True:
    meu = MusicDBIO()
    mmeDF = meu.getData()
    deezerMatchedArtistsData = mmeDF[mmeDF["DeezerAPI"].notna()][["ArtistName", "DeezerAPI"]]
    deezerMatchedArtists = deezerMatchedArtistsData["ArtistName"].copy(deep=True)
    deezerMatchedArtists.index = deezerMatchedArtistsData["DeezerAPI"]
    artistRelatedToGet = Series({artistID: artistName for artistID,artistName in deezerMatchedArtists.iteritems() if knownRelatedArtists.get(artistID) is None})
    print("{0} Search Results".format(db))
    print("   Known Master Related Data:   {0}".format(len(deezerMatchedArtists)))
    print("   Known Local Related Names:   {0}".format(len(knownRelatedArtists)))
    print("   Artist Names To Get:         {0}".format(len(artistRelatedToGet)))

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
#tt = TermTime("today", "7:36pm")
tt = TermTime("today", "10:15pm")
maxN = 50000000

n  = 0
searchedForLocalRelated         = localRelated.get()
searchedForLocalRelatedData     = localRelatedData.get()
searchedForLocalArtistsInfo     = localArtistsInfo.get()
searchedForLocalArtistsInfoData = localArtistsInfoData.get()
searchedForErrors               = errors.get()
for i,(artistID,artistName) in enumerate(artistRelatedToGet.iteritems()):
    if searchedForLocalRelated.get(artistID) is not None:
        continue

    response = apiio.getArtistRelatedData(artistName=artistName, artistID=artistID)
    if response is None or len(response) == 0:
        if response is None:
            print("Error ==> {0}".format((artistID,artistName)))
            searchedForErrors[artistID] = artistName
            searchedForLocalRelated[artistID] = artistName
            apiio.sleep(3.5)
        else:
            searchedForLocalRelated[artistID] = artistName
            apiio.sleep(1.5)
        continue
    
    searchedForLocalRelated[artistID]     = artistName
    searchedForLocalRelatedData[artistID] = {str(rec['id']): rec['name'] for rec in response}
    for record in response:
        recID = str(record['id'])
        if searchedForLocalArtistsInfo.get(recID) is None:
            searchedForLocalArtistsInfo[recID]     = artistName
            searchedForLocalArtistsInfoData[recID] = record
    apiio.sleep(1.5)
    n += 1
        
    if n % 25 == 0:
        apiio.sleep(2.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist (Related) IDs".format(len(searchedForLocalRelated), db))
        localRelated.save(data=searchedForLocalRelated)
        localRelatedData.save(data=searchedForLocalRelatedData)
        print("Saving {0} (New={1}) {2} Searched For Artist (Info) IDs".format(len(searchedForLocalArtistsInfo), len(searchedForLocalArtistsInfoData), db))
        localArtistsInfo.save(data=searchedForLocalArtistsInfo)
        localArtistsInfoData.save(data=searchedForLocalArtistsInfoData)
        if len(searchedForErrors) > 0:
            errors.save(data=searchedForErrors)
        print("="*150)
        apiio.sleep(2.5)
        if tt.isFinished():
            break
    
    if n >= maxN:
        print("Breaking after {0} downloads...".format(maxN))
        break
    
ts.stop()
print("Saving {0} {1} Searched For Artist (Related) IDs".format(len(searchedForLocalRelated), db))
localRelated.save(data=searchedForLocalRelated)
localRelatedData.save(data=searchedForLocalRelatedData)
print("Saving {0} {1} Searched For Artist (Info) IDs".format(len(searchedForLocalArtistsInfo), db))
localArtistsInfo.save(data=searchedForLocalArtistsInfo)
localArtistsInfoData.save(data=searchedForLocalArtistsInfoData)
if len(searchedForErrors) > 0:
    errors.save(data=searchedForErrors)

## Save Results

In [ ]:
from pandas import DataFrame, Series, concat
from listUtils import getFlatList

def getArtistNamesDataFrame(laid):
    df = None
    if isinstance(laid,dict) and len(laid) > 0:
        df = DataFrame(laid.values())
    return df
        
def getResponseDataFrame(laid):
    df = getArtistNamesDataFrame(laid)
    if not isinstance(df,DataFrame):
        return None
    df.index = df['id']
    df.drop(['id'], axis=1, inplace=True)
    return df

In [ ]:
laid = localArtistsInfoData.get()
df  = getResponseDataFrame(laid)
if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = mio.data.getArtistsInfoData()    
    if isinstance(searchDF,DataFrame):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = concat([searchDF,df])
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Artists".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Artists".format(searchDF.shape[0]))
    searchDF['fans'] = searchDF['fans'].astype(int)
    searchDF = searchDF.sort_values(by='fans', ascending=False)
    print("Found {0} Unique + Sorted Artists".format(searchDF.shape[0]))
    print("Saving Data")
    mio.data.saveArtistsInfoData(data=searchDF)
else:
    print("No new artists")

In [ ]:
df = localRelatedData.get()
if isinstance(df,dict):
    print("Found {0} New Artists".format(len(df)))
    searchDF = mio.data.getRelatedArtistsData()
    if isinstance(searchDF,Series):
        print("Found {0} Previous Artists".format(searchDF.shape[0]))
        searchDF = searchDF.append(Series(df))
    else:
        print("Found 0 Previous Artists")
        searchDF = df
    print("Found {0} Total Artists".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Artists".format(searchDF.shape[0]))
    print("Saving Data")
    mio.data.saveRelatedArtistsData(data=searchDF)

In [ ]:
localRelatedData.save(data={})
localArtistsInfoData.save(data={})

# Extra

## Download Data By Artist IDs

In [ ]:
print("{0} Search Results".format(db))
print("   Known Summary IDs:    {0}".format(len(knownArtists)))
artistIDsToGet = knownArtists[knownArtists.str.len() < 1].index
print("   Artists IDs To Get:   {0}".format(len(artistIDsToGet)))
artistIDsToGet = artistIDsToGet[~artistIDsToGet.isin(searchedForArtistIDs.keys())]
print("   Artists IDs To Get:   {0}".format(len(artistIDsToGet)))

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} ArtistIDs".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
tt = TermTime("today", "9:15pm")
n  = 0
searchedForArtistIDsData = getSearchedForLocalArtistIDsData()
searchedForArtistIDs = getSearchedForLocalArtistIDs()
searchedForErrors = getSearchedForErrors()
for i,dbID in enumerate(artistIDsToGet):
    if searchedForArtistIDs.get(dbID) is not None:
        continue
    if searchedForErrors.get(dbID) is not None:
        continue

    response = apiio.getArtistIDLookupResults(artistID=dbID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((dbID,artistName)))
        searchedForErrors[dbID] = "?"
        saveSearchedForErrors(searchedForErrors)
        apiio.sleep(5)
        continue
        
    searchedForArtistIDsData[dbID] = response
    searchedForArtistIDs[dbID] = True
    apiio.sleep(2.5)
    n += 1
        
    if n % 25 == 0:
        apiio.sleep(7.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Artist IDs".format(len(searchedForArtistIDs), db))
        saveSearchedForLocalArtistIDs(searchedForArtistIDs)
        saveSearchedForLocalArtistIDsData(searchedForArtistIDsData)
        print("="*150)
        apiio.sleep(7.5)
        if tt.isFinished():
            break
    
ts.stop()
print("Saving {0} {1} Searched For Artist IDs".format(len(searchedForArtistIDs), db))
saveSearchedForLocalArtistIDs(searchedForArtistIDs)
saveSearchedForLocalArtistIDsData(searchedForArtistIDsData)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Searched For Errors".format(len(searchedForErrors), db))
    saveSearchedForErrors(searchedForErrors)

In [ ]:
from pandas import DataFrame, Series, concat

def getFollowers(x):
    if isinstance(x, dict):
        return x.get('total', -1)
    return -1

def getArtistIDsDataFrame(aids):
    df = None
    if isinstance(aids,dict):
        df = DataFrame([v for k,v in aids.items()])[0].apply(Series) if len(aids) > 0 else None
    return df
        
def getResponseDataFrame(aids):
    df = getArtistIDsDataFrame(aids)
    if not isinstance(df,DataFrame):
        return None
    #df = DataFrame(response)
    df["followers"] = df["followers"].apply(getFollowers)
    df.index = df['sid']
    df.drop(['sid'], axis=1, inplace=True)
    return df

In [ ]:
aids = getSearchedForLocalArtistIDsData()
df   = getResponseDataFrame(aids)
if isinstance(df,DataFrame):
    print("Found {0} New Artists".format(df.shape[0]))
    searchDF = mio.data.getSearchArtistData()
    print("Found {0} Previous Artists".format(searchDF.shape[0]))
    searchDF = concat([searchDF,df])
    print("Found {0} Total Artists".format(searchDF.shape[0]))
    searchDF = searchDF[~searchDF.index.duplicated(keep='first')]
    print("Found {0} Unique Artists".format(searchDF.shape[0]))
    searchDF = searchDF.sort_values(by='popularity', ascending=False)
    print("Found {0} Unique + Sorted Artists".format(searchDF.shape[0]))
    print("Saving Data")
    mio.data.saveSearchArtistData(data=searchDF)
else:
    print("No new artists")

In [ ]:
saveSearchedForLocalArtistIDsData({})

# Move Local

In [ ]:
from mdblib.spotify import MusicDBIO
mio = MusicDBIO()

In [ ]:
from mdblib.spotify import moveLocalFiles

In [ ]:
moveLocalFiles()

# Download Album Data

## Find Albums To Get

In [ ]:
print("{0} Search Results".format(db))
print("   Known Summary IDs:    {0}".format(len(knownArtists)))
artistNamesToGet = knownArtists[~knownArtists.index.isin(searchedForAlbums.keys())]
artistNamesToGet = artistNamesToGet #.sample(frac=1)
print("   Artists IDs To Get:   {0}".format(len(artistNamesToGet)))

In [ ]:
from timeutils import Timestat, TermTime

ts = Timestat("Getting {0} Albums".format(db))
#tt = TermTime("tomorrow", "7:00am")
#tt = TermTime("tomorrow", "11:00am")
tt = TermTime("today", "4:00pm")
n  = 0
searchedForAlbums = getSearchedForLocalAlbums()
searchedForErrors = getSearchedForErrors()
for i,(dbID,artistName) in enumerate(artistNamesToGet.iteritems()):    
    if searchedForAlbums.get(dbID) is not None:
        continue
    if searchedForErrors.get(dbID) is not None:
        continue

    modVal=mio.mv.get(dbID)
    if mio.data.getRawArtistAlbumFilename(modVal, dbID).exists():
        searchedForAlbums[dbID] = artistName
        continue
    
    response = apiio.getArtistAlbums(artistName=artistName, artistID=dbID)
    if response is None or len(response) == 0:
        print("Error ==> {0}".format((dbID,artistName)))
        searchedForErrors[dbID] = artistName
        saveSearchedForErrors(searchedForErrors)
        apiio.sleep(5)
        continue
        
    mio.data.saveRawArtistAlbumData(data=response, modval=modVal, dbID=dbID)        
    searchedForAlbums[dbID] = artistName
    apiio.sleep(7.5)
    n += 1
        
    if n % 5 == 0:
        apiio.sleep(7.5)
        print("="*150)
        ts.update(n=n)
        print("Saving {0} {1} Searched For Albums".format(len(searchedForAlbums), db))
        saveSearchedForLocalAlbums(searchedForAlbums)
        print("="*150)
        apiio.sleep(7.5)
        if tt.isFinished():
            break
    
    if n % 25 == 0:
        apiio.sleep(30.0)
    
ts.stop()
print("Saving {0} {1} Searched For Albums".format(len(searchedForAlbums), db))
saveSearchedForLocalAlbums(searchedForAlbums)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Searched For Errors".format(len(searchedForErrors), db))
    saveSearchedForErrors(searchedForErrors)

In [ ]:
print("Saving {0} {1} Searched For Albums".format(len(searchedForAlbums), db))
saveSearchedForLocalAlbums(searchedForAlbums)
if len(searchedForErrors) > 0:
    print("Saving {0} {1} Searched For Errors".format(len(searchedForErrors), db))
    saveSearchedForErrors(searchedForErrors)

# Move Local Files

In [ ]:
from mdblib import spotify
spotify.moveLocalFiles()

# Parse What We Got

In [ ]:
%autoreload
from mdbutils import poolIO
mio = discogs.MusicDBIO(debug=False)
poolIO(parseFunction=mio.prd.parseAlbumData, modVals=range(100))